In [0]:
from pyspark.sql.functions import col, current_timestamp

# 1. Configure paths
folder_path = "abfss://retail@ameradam123.dfs.core.windows.net/bronze/AmerEgeh/Azure-Data-Engineer-Project/refs/heads/main/"
checkpoint_path = "abfss://retail@ameradam123.dfs.core.windows.net/bronze/_checkpoints/raw_sales"
schema_path = "abfss://retail@ameradam123.dfs.core.windows.net/bronze/_schemas/raw_sales"

# 2. Set up the Auto Loader stream to read incoming Parquet formats
df_stream = spark.readStream \
  .format("cloudFiles") \
  .option("cloudFiles.format", "parquet") \
  .option("cloudFiles.schemaLocation", schema_path) \
  .load(folder_path)

# 3. Inject modern Unity Catalog audit columns using the hidden _metadata column
df_stream = df_stream.withColumn("bronze_source_file", col("_metadata.file_path")) \
                     .withColumn("bronze_ingestion_time", current_timestamp())

# 4. Stream write out to your Unity Catalog Table in Append-Only mode
query = df_stream.writeStream \
  .format("delta") \
  .option("checkpointLocation", checkpoint_path) \
  .outputMode("append") \
  .trigger(availableNow=True) \
  .toTable("retail_project_dev.bronze.raw_sales")

# Force the notebook to wait until the streaming micro-batch completes fully
query.awaitTermination()
print("Auto Loader complete! Incremental data safely appended to Bronze catalog.")